# HGP：矩阵、逻辑算子和距离证明
对应论文 `sec:HGP`、`eq:HGP`。所有秩均在 GF(2) 上计算。

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "qldpc_generator").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
import numpy as np
from qldpc_generator.algebra import x, y


统一接口返回两个“检查为行、量子比特为列”的矩阵。$$H_XH_Z^{\mathsf T}=0,\quad K=N-\mathrm{rank}_2H_X-\mathrm{rank}_2H_Z.\tag{1}$$

In [2]:
from qldpc_generator import hgp
from qldpc_generator.certificates import hgp_distance_witness
from qldpc_generator.representations import tanner_edges, tanner_matrix
H=np.array([[1,1,0],[0,1,1]],dtype=np.uint8)
code=hgp(H,H)
assert code.parameters()['k']==1
assert np.array_equal(tanner_matrix(code.hx.shape,tanner_edges(code.hx)),code.hx)
code.parameters()

{'n': 13,
 'k': 1,
 'rank_x': 6,
 'rank_z': 6,
 'checks_x': 6,
 'checks_z': 6,
 'max_row_weight': 4,
 'max_column_weight': 2}

选择最小经典码字 $c$ 和不在行空间中的 $e_j$，构造 $$z=(0,c\otimes e_j).\tag{2}$$ 商映射消灭所有 Z 稳定子的第二块，却不消灭这个候选。函数同时核查这些条件；该证书直接验证上界。

In [3]:
cert=hgp_distance_witness(H)
assert code.is_logical(cert['logical'],'Z')
print('coordinate:', cert['coordinate'], 'logical support:', np.flatnonzero(cert['logical']).tolist())
print('upper bound:', cert['upper_bound'])
result=code.distance(max_weight=3)
assert result['status']=='exact' and result['value']==3
result

coordinate: 0 logical support: [4, 7, 10]
upper bound: 3


{'status': 'exact',
 'value': 3,
 'sectors': {'X': {'status': 'exact',
   'value': 3,
   'lower_bound': 3,
   'witness': [4, 5, 6],
   'subsets': 57},
  'Z': {'status': 'exact',
   'value': 3,
   'lower_bound': 3,
   'witness': [4, 7, 10],
   'subsets': 59}}}

In [4]:
LX,LZ=code.logical_operators()
assert np.array_equal(LX @ LZ.T % 2,np.eye(code.k,dtype=int))
print('LX=',LX.tolist(), 'LZ=',LZ.tolist())

LX= [[0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0]] LZ= [[0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0]]


小实例可穷尽验证距离 3。一般 HGP 渐近距离结论依赖论文引用的定理及输入码性质，不能由这个 notebook 证明。旧类接口的兼容约定见 README。